In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

In [ ]:
def read_dgrid(filename):
    dgrid=np.zeros(3)
    tmp=['','','']
    with open(filename) as cubefile :
        for _ in range(3):
            next(cubefile)
        for i in range(3):
            _,tmp[0],tmp[1],tmp[2]=cubefile.readline().split()
            dgrid[i]=float(tmp[i])
    return dgrid

def read_n(filename):
    n=np.zeros(3,dtype='int')
    with open(filename) as cubefile :
        for _ in range(3):
            next(cubefile)
        for i in range(3):
            itmp,_,_,_=cubefile.readline().split()
            n[i]=int(itmp)
    return n

def read_nat(filename):
    nat=0
    with open(filename) as cubefile :
        for _ in range(2):
            next(cubefile)
        itmp,_,_,_=cubefile.readline().split()
        nat=int(itmp)
    return nat

def read_origin(filename):
    tmp=['','','']
    with open(filename) as cubefile :
        for _ in range(2):
            next(cubefile)
        itmp,tmp[0],tmp[1],tmp[2]=cubefile.readline().split()
        origin = np.array(tmp, dtype=np.float32)
    return origin

def read_data(filename):
    nat=read_nat(filename)
    n=read_n(filename)
    data=np.zeros(n[0]*n[1]*n[2])
    with open(filename) as cubefile :
        i=0 
        for _ in range(nat+6):
            next(cubefile)
        for line in cubefile:
            data[i:i+6]=[ float(s) for s in line.split()]
            i+=6
    return data.reshape((n[0],n[1],n[2]))

In [ ]:
def gen_gridr(n,dgrid,origin,delta=1.e-5):
    gridr=np.zeros((3,n[0],n[1],n[2]))
    x=np.arange(0.,dgrid[0]*n[0]-delta,dgrid[0])+origin[0]
    y=np.arange(0.,dgrid[1]*n[1]-delta,dgrid[1])+origin[1]
    z=np.arange(0.,dgrid[2]*n[2]-delta,dgrid[2])+origin[2]
    gridr[0],gridr[1],gridr[2]=np.meshgrid(x,y,z,indexing='ij')
    return gridr

In [ ]:
cubefile = 'cubefiles/fukui_negative_defect.cube'

In [ ]:
print(read_dgrid(cubefile))
print(read_n(cubefile))
print(read_nat(cubefile))

In [ ]:
dgrid=read_dgrid(cubefile)
n = read_n(cubefile)
cell = dgrid*n # large cell
origin=read_origin(cubefile)
print(cell)
print(origin)

In [ ]:
data=read_data(cubefile) 

In [ ]:
x=np.arange(0.,dgrid[0]*n[0],dgrid[0])+origin[0]
print(x.shape)

In [ ]:
dgrid[0]*n[0]

In [ ]:
gridr=gen_gridr(n,dgrid,origin)

In [ ]:
data.shape

In [ ]:
def cube2line(dgrid,n,r,data,center,axis):
    icenter = np.rint(center/dgrid).astype('int')
    icenter = icenter - n * np.trunc(icenter//n).astype('int') 
    if axis == 0 :
        axis = r[0,:,icenter[1],icenter[2]]
        value = data[:,icenter[1],icenter[2]]
    elif axis == 1 :
        axis = r[1,icenter[0],:,icenter[2]]
        value = data[icenter[0],:,icenter[2]]
    elif axis == 2 :
        axis = r[2,icenter[0],icenter[1],:]
        value = data[icenter[0],icenter[1],:]
    plt.plot(axis,value)

def cubeplanaraverage(dgrid,n,r,data,center,axis):
    icenter = np.rint(center/dgrid).astype('int')
    icenter = icenter - n * np.trunc(icenter//n).astype('int') 
    if axis == 0 :
        axis = r[0,:,icenter[1],icenter[2]]
        value = np.mean(data,(1,2))
    elif axis == 1 :
        axis = r[1,icenter[0],:,icenter[2]]
        value = np.mean(data,(0,2))
    elif axis == 2 :
        axis = r[2,icenter[0],icenter[1],:]
        value = np.mean(data,(0,1))
    plt.plot(axis,value)

In [ ]:
center=np.array([6.63108,1.96327,2.54474])
cubeplanaraverage(dgrid,n,gridr,data,center,2)

In [ ]:
plt.contourf(gridr[0,:,:,2],gridr[1,:,:,2],data[:,:,2])

in the following we are assuming the slab to be oriented along the z axis (index 2). We may want to be more general in the future implementation

In [ ]:
map_n = np.array([20,20],dtype='int')
map_z = 12. # assume it is constant to start
map_dgrid = cell[:2]/map_n
delta=0.00001
map_x = np.arange(0.,map_dgrid[0]*map_n[0]-delta,map_dgrid[0])
map_y = np.arange(0.,map_dgrid[1]*map_n[1]-delta,map_dgrid[1])
map_gridr = np.zeros((3,map_n[0],map_n[1]))
map_gridr[0], map_gridr[1] = np.meshgrid(map_x,map_y,indexing='ij')
map_gridr[2,:,:] = map_z

we assume we will only integrate the cubefile from the map position down

In [ ]:
r = 5. # radius of cutoff in plane
d = 5. # depth of cutoff below the system 
ntheta = 200
cyl_dgrid = np.zeros(3)
cyl_dgrid[0] = (dgrid[0]+dgrid[1])/2 # distance in points in plane
cyl_dgrid[1] = 2 * np.pi / ntheta + delta # distance in radiants in the angular direction
cyl_dgrid[2] = dgrid[2]
cyl_range = np.array([r,2*np.pi,d])
cyl_n = np.array(cyl_range/cyl_dgrid+1,dtype='int')

In [ ]:
delta=0.0001
cyl_r = np.arange(0,cyl_range[0]-delta,cyl_dgrid[0])
cyl_theta = np.arange(0,cyl_range[1]-delta,cyl_dgrid[1])
cyl_z = np.arange(0,cyl_range[2]-delta,cyl_dgrid[2])
cyl_gridc = np.zeros((2,cyl_n[0],cyl_n[1]))
cyl_gridc[0],cyl_gridc[1] = np.meshgrid(cyl_r,cyl_theta,indexing='ij')

In [ ]:
cyl_gridr = np.zeros((2,cyl_n[0],cyl_n[1]))
cyl_gridr[0,:,:] = cyl_gridc[0,:,:] * np.cos(cyl_gridc[1,:,:])
cyl_gridr[1,:,:] = cyl_gridc[0,:,:] * np.sin(cyl_gridc[1,:,:])

In [ ]:
cyl_z

In [ ]:
map_function = np.zeros(map_n)
#data[:,:,:]=1.
invcell = 1/cell[:2]
for z in cyl_z :
      zlayer = center[2] - z
      iz = np.rint(zlayer/dgrid[2]).astype('int')
      if iz < 0 : iz += n[2]
#      plt.subplot(2,2,2)
#      plt.axis('equal')
#      plt.contourf(gridr[0,:,:,iz], gridr[1,:,:,iz], data[:,:,iz])
#      plt.subplot(2,2,4)
#      plt.plot(gridr[0,:,0,iz],data[:,0,iz])
      spline = scipy.interpolate.RectBivariateSpline(gridr[0,:,0,iz], gridr[1,0,:,iz], data[:,:,iz], bbox=[None, None, None, None], kx=3, ky=3, s=0)# 
      for i in range(map_n[0]):
            for j in range(map_n[1]): 
                  center = map_gridr[:,i,j] 
                  inplane = cyl_gridr[:,:,:] + center[:2,None,None]
                  scaled = np.einsum('ijk,i->ijk',inplane,invcell)
                  scaled = scaled - np.floor(scaled)
                  inplane = np.einsum('ijk,i->ijk',scaled,cell[:2])
                  interpolated_data = spline.ev(inplane[0,:,:],inplane[1,:,:])
#                  plt.subplot(1,2,1)
#                  plt.contourf(inplane[0,:,:],inplane[1,:,:],interpolated_data)
#                  plt.scatter(inplane[0,:,:],inplane[1,:,:])
                  map_function[i,j] += np.einsum('ij,ij->',interpolated_data,cyl_gridc[0,:,:])
map_function = map_function  * cyl_dgrid[0] * cyl_dgrid[1] * cyl_dgrid[2]
plt.subplot(2,2,1)
plt.contourf(map_gridr[0,:,:],map_gridr[1,:,:],map_function)
plt.axis('equal')
plt.subplot(2,2,3)
plt.plot(map_gridr[0,:,0],map_function[:,0]/np.pi/cyl_range[0]**2/cyl_range[2])


In [ ]:
map_function = np.zeros(map_n)
#data[:,:,:]=1.
invcell = 1/cell[:2]
for z in [0] : # cyl_z :
      zlayer = center[2] - z
      iz = np.rint(zlayer/dgrid[2]).astype('int')
      if iz < 0 : iz += n[2]
      plt.subplot(1,2,1)
      plt.axis('equal')
      plt.contourf(gridr[0,:,:,iz], gridr[1,:,:,iz], data[:,:,iz])
      plt.scatter(map_gridr[0,4,5],map_gridr[1,4,5])
#      plt.subplot(2,2,4)
#      plt.plot(gridr[0,:,0,iz],data[:,0,iz])
      spline = scipy.interpolate.RectBivariateSpline(gridr[0,:,0,iz], gridr[1,0,:,iz], data[:,:,iz], bbox=[None, None, None, None], kx=3, ky=3, s=0)# 
      f_of_theta = np.zeros(cyl_n[1])
      for i in [4] : # range(map_n[0]):
            for j in [5] : # range(map_n[1]): 
                  center = map_gridr[:,i,j] 
                  inplane = cyl_gridr[:,:,:] + center[:2,None,None]
                  scaled = np.einsum('ijk,i->ijk',inplane,invcell)
                  scaled = scaled - np.floor(scaled)
                  inplane = np.einsum('ijk,i->ijk',scaled,cell[:2])
                  interpolated_data = spline.ev(inplane[0,:,:],inplane[1,:,:])
                  f_of_theta = np.einsum('ij,ij->j',interpolated_data,cyl_gridc[0,:,:]) * cyl_dgrid[0]
plt.subplot(1,2,2)
plt.plot(cyl_gridc[1,0,:],f_of_theta)


In [ ]:
map_function_0 = np.zeros(map_n)
map_function_1 = np.zeros(map_n)
map_function_2 = np.zeros(map_n)
map_function_3 = np.zeros(map_n)
map_function_4 = np.zeros(map_n)
map_function_6 = np.zeros(map_n)
#data[:,:,:]=1.
invcell = 1/cell[:2]
for z in cyl_z :
      zlayer = center[2] - z
      iz = np.rint(zlayer/dgrid[2]).astype('int')
      if iz < 0 : iz += n[2]
      spline = scipy.interpolate.RectBivariateSpline(gridr[0,:,0,iz], gridr[1,0,:,iz], data[:,:,iz], bbox=[None, None, None, None], kx=3, ky=3, s=0)# 
      f_of_theta = np.zeros(cyl_n[1])
      for i in range(map_n[0]):
            for j in range(map_n[1]): 
                  center = map_gridr[:,i,j] 
                  inplane = cyl_gridr[:,:,:] + center[:2,None,None]
                  scaled = np.einsum('ijk,i->ijk',inplane,invcell)
                  scaled = scaled - np.floor(scaled)
                  inplane = np.einsum('ijk,i->ijk',scaled,cell[:2])
                  interpolated_data = spline.ev(inplane[0,:,:],inplane[1,:,:])
                  f_of_theta = np.einsum('ij,ij->j',interpolated_data,cyl_gridc[0,:,:]) * cyl_dgrid[0]
                  ff = scipy.fft.fft(f_of_theta)
                  map_function_0[i,j] += 2.0/ntheta * np.abs(ff[0])
                  map_function_1[i,j] += 2.0/ntheta * np.abs(ff[1])
                  map_function_2[i,j] += 2.0/ntheta * np.abs(ff[2])
                  map_function_3[i,j] += 2.0/ntheta * np.abs(ff[3])
                  map_function_4[i,j] += 2.0/ntheta * np.abs(ff[4])
                  map_function_6[i,j] += 2.0/ntheta * np.abs(ff[6])
plt.subplot(2,3,1)
plt.axis('equal')
plt.contourf(map_gridr[0,:,:], map_gridr[1,:,:], map_function_0[:,:])
plt.subplot(2,3,2)
plt.axis('equal')
plt.contourf(map_gridr[0,:,:], map_gridr[1,:,:], map_function_1[:,:])
plt.subplot(2,3,3)
plt.axis('equal')
plt.contourf(map_gridr[0,:,:], map_gridr[1,:,:], map_function_2[:,:])
plt.subplot(2,3,4)
plt.axis('equal')
plt.contourf(map_gridr[0,:,:], map_gridr[1,:,:], map_function_3[:,:])
plt.subplot(2,3,5)
plt.axis('equal')
plt.contourf(map_gridr[0,:,:], map_gridr[1,:,:], map_function_4[:,:])
plt.subplot(2,3,6)
plt.axis('equal')
plt.contourf(map_gridr[0,:,:], map_gridr[1,:,:], map_function_6[:,:])

# Understanding FFT in Scipy

In [ ]:
N = ntheta
# sample spacing
T = 2 * np.pi * 1.0 / ntheta
x = np.linspace(0.0, N*T, N, endpoint=False)
y = -1.45e-6 - 0.38e-6 * np.sin(3.0 * x)
plt.plot(x,y)
plt.plot(x,f_of_theta)

In [ ]:
yf = scipy.fft.fft(y)
xf = scipy.fft.fftfreq(N, T)[:N//2]
plt.plot(2*np.pi*xf, 2.0/N * np.abs(yf[0:N//2]),'o-')
plt.xlim(0,20)
plt.grid()
plt.show()